# 3D Scan Pipeline - Part 2: Training

**Objective:** Train the Gaussian Splatting model using Taichi and Export to .splat.
**Input:** Use the output from Part 1 (`3d_scan_data_part1.zip`) as a **Kaggle Dataset**.

**How to use on Kaggle:**
1. In Part 1, download `3d_scan_data_part1.zip`.
2. Creating a specific Dataset in Kaggle with this file.
3. Add your new Dataset to this notebook (Part 2).

**Environment:** **GPU REQUIRED (T4 or better)**.

In [ ]:
import os
import sys

print("⏳ Setting up Environment (Part 2)...")

# 1. Clone Repo (Only if local project not found)
if not os.path.exists("3DSCAN"):
    !git clone https://github.com/PRIDA-TAKON/3DSCAN.git
    if os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")
else:
    print("📂 Project folder found. Using local version.")
    if os.path.basename(os.getcwd()) != "3DSCAN" and os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")

# 2. Install Dependencies
print("⏳ Installing Dependencies...")
!pip install --upgrade pip
!pip install --upgrade numba scipy pandas scikit-learn opencv-python opencv-python-headless opencv-contrib-python matplotlib pillow plyfile tqdm roma
!pip install taichi

# Install Taichi Splatting (Wanmeihuali Version)
if os.path.exists("taichi_3d_gaussian_splatting"):
    print("📂 taichi_3d_gaussian_splatting found. Skipping clone.")
else:
    !git clone --depth 1 https://github.com/wanmeihuali/taichi_3d_gaussian_splatting.git

# Standard install without forcing numpy version (Using Kaggle Default)
!pip install -r taichi_3d_gaussian_splatting/requirements.txt
!pip install ./taichi_3d_gaussian_splatting

# Verify Environment
try:
    import numpy
    print(f"✅ Setup Complete. NumPy Version: {numpy.__version__}")
except Exception as e:
    print(f"⚠️ Error checking numpy: {e}")

In [ ]:
print("=== Import Data from Kaggle Dataset ===")
import glob
import zipfile
import shutil

# Reset working_data to avoid conflicts
if os.path.exists("working_data"):
    shutil.rmtree("working_data", ignore_errors=True)
os.makedirs("working_data/3d_scan", exist_ok=True)

# Look for the dataset zip file in /kaggle/input (standard dataset path)
search_paths = ["/kaggle/input", "input", "."]
zip_path = None

for path in search_paths:
    candidates = glob.glob(f"{path}/**/*.zip", recursive=True)
    for c in candidates:
        if "3d_scan_data_part1" in c or "3d_scan_output" in c:
            zip_path = c
            break
    if zip_path: break

if not zip_path:
    # Fallback: check if the folder structure already exists (unzipped dataset)
    # Maybe the user uploaded the folder structure directly
    folder_candidates = glob.glob("/kaggle/input/**/sparse/0", recursive=True)
    if folder_candidates:
        print("📂 Found unzipped dataset structure. Copying...")
        # Identify the root '3d_scan' folder from the found path
        # e.g. /kaggle/input/my-dataset/working_data/3d_scan/sparse/0 -> root is .../working_data/3d_scan
        src_sparse = os.path.dirname(folder_candidates[0]) # .../sparse
        src_root = os.path.dirname(src_sparse) # .../3d_scan
        
        # Copy everything to local working_data
        # copying from read-only input to writable working dir
        print(f"   Copying from {src_root} to working_data/3d_scan...")
        import distutils.dir_util
        distutils.dir_util.copy_tree(src_root, "working_data/3d_scan")
        print("✅ Data copied successfully.")
    else:
        print("❌ No data found! Please add the '3d_scan_data_part1' dataset.")
else:
    print(f"📦 Found data zip: {zip_path}")
    print("⏳ Extracting to working_data...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("✅ Extraction Complete.")
    
# Verify structure
if os.path.exists("working_data/3d_scan/sparse"):
    print("✅ Data Validation: Sparse model found.")
else:
    print("⚠️ Warning: 'sparse' folder not found in working_data/3d_scan. Training might fail.")

In [ ]:
print("=== STEP 3: Train Taichi Splatting ===")
# Ensure the folder structure is correct
if not os.path.exists("working_data/3d_scan"):
    print("❌ working_data/3d_scan not found. Check dataset.")
else:
    !python scripts/step3_train_splatting.py --project_path "working_data/3d_scan" --output_path "outputs/3d_scan/taichi_splatting"

In [ ]:
print("=== STEP 4: Export ===")
!python scripts/step4_export.py --input_parquet "outputs/3d_scan/taichi_splatting/model.parquet" --output_splat "outputs/3d_scan/taichi_splatting/model.splat"

In [ ]:
print("=== Compress Final Model for Download ===")
output_model_zip = "3d_splat_model.zip"

if os.path.exists("outputs/3d_scan/taichi_splatting"):
    !zip -r {output_model_zip} outputs/3d_scan/taichi_splatting
    
    from IPython.display import FileLink
    display(FileLink(output_model_zip))
else:
    print("❌ No output model found.")